# Seção 9 (propriedades e campos calculados) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`03_computed_fields`](03_computed_fields.ipynb) | campo calculado sem anotação de retorno ainda gera `PydanticUserError` |
| [`05_projeto_solucao_instrutor`](05_projeto_solucao_instrutor.ipynb) | `KeyError` no `model_dump()` sem país; código de país cacheado desatualizado |

## Novas

**1. Campo calculado roda em **todo** dump.**

In [1]:
from functools import cached_property
from pydantic import BaseModel, ConfigDict, TypeAdapter, computed_field

price_lookups = {"count": 0}


class Item(BaseModel):
    sku: str

    @computed_field
    @property
    def price(self) -> float:
        price_lookups["count"] += 1                     # imagine uma consulta ao banco
        return 10.0


catalog = [Item(sku=str(n)) for n in range(100)]
items_adapter = TypeAdapter(list[Item])
items_adapter.dump_python(catalog)
items_adapter.dump_json(catalog)
print(f"consultas para 2 dumps de 100 itens: {price_lookups['count']}")

consultas para 2 dumps de 100 itens: 200


Uma listagem paginada que serializa para log **e** para a resposta dobra as consultas. Cálculo caro: `cached_property` (com o
cuidado de mutabilidade da seção), ou calcular em lote antes e passar como campo comum.

**2. `model_copy(update=...)` copia o cache junto.**

In [2]:
class Circle(BaseModel):
    model_config = ConfigDict(frozen=True)
    radius: float

    @cached_property
    def area(self) -> float:
        return 3.14159 * self.radius ** 2


small = Circle(radius=1)
print(f"small.area: {small.area}")
big = small.model_copy(update={"radius": 10})
print(f"big.radius: {big.radius}")
print(f"big.area (deveria ser 314.159): {big.area}")

small.area: 3.14159
big.radius: 10
big.area (deveria ser 314.159): 3.14159


`frozen` deixa o cache "seguro" para atribuição, mas `model_copy` copia o `__dict__` — inclusive o valor cacheado — e o `update`
não passa por validação nem invalida nada. "Alterar" um modelo frozen com cache: `Circle(**small.model_dump(), radius=10)` ou
`Circle.model_validate({...})`.

**3. Setter de propriedade grava sem validar.**

In [3]:
class Sensor(BaseModel):
    celsius: float

    @property
    def reading(self) -> float:
        return self.celsius

    @reading.setter
    def reading(self, value) -> None:
        self.celsius = value                            # sem validate_assignment: nada é checado


sensor = Sensor(celsius=21.5)
sensor.reading = "quente"
print(f"sensor.celsius: {sensor.celsius!r}")

sensor.celsius: 'quente'


Um `float` passou a guardar texto, e o erro aparece longe (num cálculo, num dump para JSON tipado). Com setters, ligue
`validate_assignment=True` — o `self.celsius = ...` do setter passa a ser validado.

## Veja também

- [`dataclass_factory/04_politicas_e_presets`](../../../dataclass_factory/04_politicas_e_presets.ipynb) (caveats): `model_copy(update=...)` pula validação.